# 05 — Assemble the v2 pipeline and evaluate it

Loads the two new detectors, runs the tooth -> disease -> association pipeline
on the **locked 50-image official validation set**, and scores it with the same
harness that measured v1. The numbers are therefore directly comparable.

Run this locally after downloading `tooth_best.pt` and `disease_best.pt` into
`dentex_week3/production/`.

In [ ]:
import json
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROD = REPO / "dentex_week3" / "production"
sys.path.insert(0, str(REPO / "dentex_week3"))

for weight in ("tooth_best.pt", "disease_best.pt"):
    p = PROD / weight
    print(f"{weight:20s} {'OK' if p.exists() else 'MISSING'} "
          f"{(p.stat().st_size / 1e6 if p.exists() else 0):.1f} MB")
assert (PROD / "tooth_best.pt").exists() and (PROD / "disease_best.pt").exists(), \
    "download the Kaggle outputs into dentex_week3/production/ first"

In [ ]:
import importlib
import inference_pipeline as pipe

importlib.reload(pipe)
print("tooth model  :", pipe.CONFIG["tooth_model"], "imgsz", pipe.TOOTH_IMGSZ, "conf", pipe.TOOTH_CONF)
print("disease model:", pipe.CONFIG["disease_model"], "imgsz", pipe.DISEASE_IMGSZ, "conf", pipe.DISEASE_CONF)

In [ ]:
# ---- smoke test on three images ----
DATA = Path("/home/sayfeldin/Projects/dental-model-research")
VAL_IMAGES = DATA / "data/raw/dentex/validation_data/quadrant_enumeration_disease/xrays"

for name in ("val_0.png", "val_1.png", "val_2.png"):
    out = pipe.run_dentex_pipeline(str(VAL_IMAGES / name))
    print(f"\n{name}: {out['num_detections']} lesions, {len(out['quadrants'])} quadrants")
    for d in out["detections"][:6]:
        print(f"  FDI {d['tooth_fdi']:>2s} {d['disease']:20s} "
              f"p={d['disease_confidence']:.2f} tooth_p={d['tooth_confidence']:.2f}")

In [ ]:
# ---- full run on the locked 50 ----
from evaluate import load_ground_truth, evaluate, render_report

GT = DATA / "data/downloads/validation_triple.json"
gt = load_ground_truth(GT)

cache = {}
for i, name in enumerate(sorted(gt), 1):
    try:
        cache[name] = {"image": name, "ok": True,
                       "result": pipe.run_dentex_pipeline(str(VAL_IMAGES / name))}
    except Exception as exc:
        cache[name] = {"image": name, "ok": False, "error": f"{type(exc).__name__}: {exc}"}
    if i % 10 == 0:
        print(f"  {i}/{len(gt)}")

failed = [n for n, r in cache.items() if not r["ok"]]
print(f"scored {len(cache) - len(failed)}/{len(gt)} | failed: {failed}")

In [ ]:
results = evaluate(cache, gt)
report = render_report(results)

out_dir = REPO / "dentex_week3" / "eval"
(out_dir / "evaluation_results_v2.json").write_text(json.dumps(results, indent=2))
(out_dir / "evaluation_report_v2.md").write_text(report)
with open(out_dir / "raw_predictions_v2.jsonl", "w") as fh:
    for name, rec in cache.items():
        fh.write(json.dumps(rec) + "\n")
print(report)

In [ ]:
# ---- v1 vs v2 ----
v1 = json.loads((REPO / "dentex_week3/eval/evaluation_results.json").read_text())

def row(label, old, new):
    print(f"{label:22s} {old:>8.1%} {new:>8.1%} {new - old:>+8.1%}")

print(f"{'metric':22s} {'v1':>8s} {'v2':>8s} {'delta':>8s}")
row("mAP50", v1["mAP50"], results["mAP50"])
row("precision", v1["detection_micro"]["precision"], results["detection_micro"]["precision"])
row("recall", v1["detection_micro"]["recall"], results["detection_micro"]["recall"])
row("disease acc", v1["label_accuracy"]["disease"]["rate"], results["label_accuracy"]["disease"]["rate"])
row("tooth acc", v1["label_accuracy"]["tooth"]["rate"], results["label_accuracy"]["tooth"]["rate"])
row("joint acc", v1["label_accuracy"]["joint"]["rate"], results["label_accuracy"]["joint"]["rate"])
print("\nper-class AP50")
for cls in results["detection_per_class"]:
    print(f"  {cls:22s} {v1['ap_per_class'][cls]['AP50']:>7.1%} -> {results['ap_per_class'][cls]['AP50']:>7.1%}")

## Verdict rules

The rebuild is promoted only if it beats v1 on the locked set:

| metric | must beat |
|---|---|
| mAP@0.5 | 49.8% |
| Caries AP@0.5 | 28.2% |
| tooth accuracy | 75.9% |
| joint accuracy | 66.4% |

If tooth accuracy improves but detection regresses (or vice versa), keep the
winning stage and retrain the other — the stages are independent by design.